# OTOCs of the memory: plots

**12/10/2026, definition of $g_2$.** The article uses $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$ (static pump, as in the EOMs).
The quantum runs of the `_12_10_2026` files (written at 17:59) set $\varepsilon_d = r\,g_2$ with this $g_2$ and $r$ in `ratio_list`,
so the heatmaps are on the regular grid $(\varepsilon_p, \varepsilon_d/g_2 = r)$ and no relabelling is needed.
(The earlier J1 runs, $\varepsilon_d = r\,J_1(\varepsilon_p)E_J\varphi_a^2\varphi_b$, needed $\varepsilon_d/g_2 = r\,J_1/\sin$; applying that to the new
data shears and halves the $\varepsilon_d/g_2$ axis.) The classical lines are from the `_12_10_2026` files, computed with the same definition.

Loads the OTOC results written by `OTOCS_poster_init_coherent.ipynb`, copied to
`classical_simulations/analysis/output/chicago_poster_2026/`, and plots them. Nothing is simulated here, so this needs neither the
cluster nor jax.

- `otoc_example_<date>.json`, `otoc_example_stable_<date>.json`: `C(t) =
  -<[x_a(t), p_a(0)]^2>` at `epsilon_0 = 0.93` (chaotic) and `0.2` (stabilized),
  `epsilon_d = 10 g2`, `n_a = 38`.
- `lyap_array_init_coherent_<date>.json`, `R_sq_array_init_coherent_<date>.json`,
  `end_otoc_array_init_coherent_<date>.json`:
  the heatmap over `epsilon_0` and `epsilon_d / g2` at `n_a = 29`, `n_b = 17`. These
  hold the rate `m / 2` and the `R^2` of the linear fit of `log C` against `kappa_b
  t`, and `C(T)`.

The JSONs hold only the values. The heatmap files are a list of rows, one per
`epsilon_0`, one entry per `epsilon_d / g2`. The example curves have no time
axis, so it is rebuilt from the curve's length, which tells the two grids apart:

- **25 points** (runs from 2026-10-12 17:59 on): adjoint equation with buffer loss. The jump operator is
  `sqrt(kappa) b` with `kappa = 2 pi / 10.4`, i.e. `kappa / 2pi` = 96 MHz, the article's `kappa_b`. The producer uses
  `n_sub = 8` points per drive period `tau = 2 pi / w_b` over `ceil(0.2 / (kappa tau)) = 3` periods, so
  `kappa_b T = 0.225` (T = 0.37 ns). It computes the times one offset `k tau / n_sub` at a time and sorts them, so the
  saved curve is on `tau / n_sub * arange(n)`. Checked against the producer: the fitted slope `m = 13.58` of the
  chaotic cat-start example is the value it printed.
  Until 13/10/2026 this notebook read these curves as 6 periods of 4 points with `kappa = pi / 10.4`. The two
  factors of 2 cancelled, so `kappa_b t`, the fits and the figures were already right.
- **49 points:** earlier adjoint-equation runs with `kappa = pi / 10.4` (jump `sqrt(kappa) b`, 48 MHz): 6 periods of
  8 points.
- **50 points:** the earlier closed-system version (also `OTOCS_poster.ipynb`).
  It uses `linspace(0, 0.2 / kappa, 50)` with `kappa = pi / 10.4`.

The two give different physics, so curves of different dates are not
comparable. `OTOCS_poster.ipynb` writes the same file names, so a run of it on
the same day overwrites these files.

If a grid changes in `OTOCS_poster_init_coherent.ipynb`, change it here too. The
checks below catch a mismatch in size, but not in values.

In [ ]:
import json
import pathlib
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import Perso_colormap
import matplotlib.pyplot as plt
import numpy as np

output_dir = pathlib.Path("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/"
                          "analysis/output/chicago_poster_2026")

# The earlier closed-system run is otoc_example{,_stable}_22_04_2026.json and
# {lyap,R_sq,end_otoc}_array_22_04_2026_bis.json.
examples_date = heatmap_date = "12_10_2026"
EXAMPLE_FILES = {"otoc_example": "otoc_example_init_coherent_12_10_2026.json",
                 "otoc_example_stable": "otoc_example_stable_init_coherent_12_10_2026.json"}
HEATMAP_FILES = {"lyap_array": "lyap_array_init_coherent_12_10_2026.json",
                 "R_sq_array": "R_sq_array_init_coherent_12_10_2026.json",
                 "end_otoc_array": "end_otoc_array_init_coherent_12_10_2026.json"}


def load_group(files):
    """{name: array} for the files {name: file name} in output_dir."""
    return {name: np.array(json.load(open(output_dir / f)), dtype=float) for name, f in files.items()}


examples = load_group(EXAMPLE_FILES)
heatmap = load_group(HEATMAP_FILES)
for files, group in ((EXAMPLE_FILES, examples), (HEATMAP_FILES, heatmap)):
    for name, arr in group.items():
        print(f"loaded {files[name]}: shape {arr.shape}, "
              f"{np.isnan(arr).sum()} failed point(s), "
              f"values in [{np.nanmin(arr):.4g}, {np.nanmax(arr):.4g}]")

In [ ]:
# Constants and grids of OTOCS_poster_init_coherent.ipynb.
w_a = 25.338776456203686
w_b = 2 * w_a
kappa = 2 * np.pi / 10.4      # buffer loss of the quantum runs, jump sqrt(kappa) b: kappa/2pi = 96 MHz, the article's kappa_b
kappa_old = np.pi / 10.4      # kappa of the runs before 2026-10-12 17:59 (49- and 50-point curves)
tau = 2 * np.pi / w_b         # drive period
N_SUB = 8                     # points per drive period in the producer


def time_axis(n_points):
    """The t grid (ns) of an example curve, from its length (see the top of this notebook)."""
    for k, label in ((kappa, "adjoint equation, buffer loss"),
                     (kappa_old, "adjoint equation, buffer loss, kappa = pi/10.4 (old)")):
        n_periods = int(np.ceil(.2 / k / tau))   # 3 for kappa, 6 for kappa_old
        if n_points == n_periods * N_SUB + 1:
            return tau / N_SUB * np.arange(n_points), f"{label}, {n_periods} periods x {N_SUB} pts"
    if n_points == 50:                            # closed system, before 2026-10-12
        return np.linspace(0, .2 / kappa_old, n_points), "closed system (old)"
    raise ValueError(f"no known time grid with {n_points} points")


# The heatmap grid (the same in OTOCS_poster.ipynb).
epsilon_0_list = list(np.concatenate((np.arange(0.2, .6, 0.07), np.arange(0.6, 1.1, 0.03))))
epsilon_0_arr = np.array(epsilon_0_list, dtype=np.float32)
ratio_list = np.arange(2, 10, 1)
n_a, n_b = 29, 17

lyap_arr = heatmap["lyap_array"]
R_sq_arr = heatmap["R_sq_array"]
end_otoc_arr = heatmap["end_otoc_array"]
for name, arr in heatmap.items():
    assert arr.shape == (len(epsilon_0_arr), len(ratio_list)), (
        f"{name}_{heatmap_date}.json is {arr.shape}, the grid here is "
        f"{(len(epsilon_0_arr), len(ratio_list))}: the two notebooks disagree on the grid")

In [ ]:
# Cell corners of the run grid, with g2 = E_J sin(eps_p) phi_a^2 phi_b (see the top of this notebook)
E_J = 37.12 * 2 * np.pi * 1.5
phi_a, phi_b = 0.11, 0.204


def g2_of(eps_p):
    """Two-photon rate of the article (static pump): E_J sin(eps_p) phi_a^2 phi_b, rad/ns."""
    return np.sin(eps_p) * E_J * phi_a**2 * phi_b


def _grid_edges(x):
    x = np.asarray(x, dtype=float)
    mid = (x[1:] + x[:-1]) / 2
    return np.concatenate(([x[0] - (mid[0] - x[0])], mid, [x[-1] + (x[-1] - mid[-1])]))


def run_mesh(eps_p_centers, ratio_centers):
    """Cell corners for pcolormesh(.., C) with C[i, j] at (eps_p_i, ratio_j), shape (n_eps + 1, n_ratio + 1):
    eps_p, g2/2pi (MHz) and eps_d/g2 = ratio."""
    E, R = np.meshgrid(_grid_edges(eps_p_centers), _grid_edges(ratio_centers), indexing="ij")
    return E, g2_of(E) / (2 * np.pi) * 1e3, R


EPS_MESH, G2_MESH, RATIO_MESH = run_mesh(epsilon_0_arr.astype(float), ratio_list)
print(f"eps_d/g2 covered: {RATIO_MESH.min():.2f} to {RATIO_MESH.max():.2f}, g2/2pi: {G2_MESH.min():.1f} to {G2_MESH.max():.1f} MHz")

## Chaotic and stabilized regimes

`C(t)` on a log scale against `kappa_b t`, with the linear fit of `log C` whose
slope `m` is the growth rate, as in `OTOCS_poster_init_coherent.ipynb`.

In [ ]:
def fit_log_otoc(t, otoc):
    """Linear fit of log C(t) against kappa_b t: slope m, intercept, R^2."""
    t_kappa = t * kappa
    log_otoc = np.log(otoc)
    (m, c), residuals, _, _, _ = np.polyfit(t_kappa, log_otoc, 1, full=True)
    R_sq = 1 - residuals.item() / np.sum((log_otoc - np.mean(log_otoc))**2)
    return m, c, R_sq


fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True, sharey=True)
for ax, (name, title) in zip(axes, (("otoc_example", r"chaotic, $\varepsilon_p = 0.93$"),
                                    ("otoc_example_stable", r"stabilized, $\varepsilon_p = 0.2$"))):
    otoc = examples[name]
    t, grid = time_axis(len(otoc))
    m, c, R_sq = fit_log_otoc(t, otoc)
    ax.plot(t * kappa, otoc, "o-", ms=3, label=r"$C(t)$")
    ax.plot(t * kappa, np.exp(m * t * kappa + c), "--", color="0.4",
            label=rf"fit: $m = {m:.3g}$, $R^2 = {R_sq:.3g}$")
    ax.set_yscale("log")
    ax.set_xlabel(r"$\kappa_b t$")
    ax.set_title(f"{title}, " + r"$n_a=38$" + f"\n{grid}, {examples_date}", fontsize=9)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=8)
    print(f"{name:<20} m = {m:.6g}   R^2 = {R_sq:.6g}   ({grid})")
axes[0].set_ylabel(r"$C(t) = -\langle[x_a(t),p_a(0)]^2\rangle$")
plt.show()

## Heatmaps over the pump amplitude and the drive strength

`epsilon_0` is not evenly spaced (steps of `0.07` up to `0.6`, then `0.03`),
so the cells are drawn at their true positions with `pcolormesh`, as in
`plot_IPR.ipynb`. `C(T)` spans more than a decade, so it is on a log colour
scale.

In [ ]:
from matplotlib.colors import LogNorm


def _edges(x):
    """Cell edges around the centers x: the midpoints, extended at both ends."""
    x = np.asarray(x, dtype=float)
    mid = (x[1:] + x[:-1]) / 2
    return np.concatenate(([x[0] - (mid[0] - x[0])], mid, [x[-1] + (x[-1] - mid[-1])]))


fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.4), constrained_layout=True)
for ax, arr, label, kw in (
        (axes[0], lyap_arr, r"$m/2$", dict(cmap=Perso_colormap)),
        (axes[1], R_sq_arr, r"$R^2$", dict(cmap=Perso_colormap, vmin=0, vmax=1)),
        (axes[2], end_otoc_arr, r"$C(T)$", dict(cmap=Perso_colormap, norm=LogNorm()))):
    im = ax.pcolormesh(EPS_MESH, RATIO_MESH, arr, **kw)
    ax.set_xlabel(r"$\varepsilon_p$" + " (rad)")
    ax.set_ylabel(r"$\varepsilon_d / g_2$")
    ax.set_title(label + r", $n_a=$" + str(n_a) + ", " + r"$n_b=$" + str(n_b)
                 + f", {heatmap_date}", fontsize=10)
    fig.colorbar(im, ax=ax, label=label)
plt.show()

## Cuts at fixed drive strength

The same data, one curve per `epsilon_d / g2`, coloured from light (`2`) to
dark (`9`). Points whose fit has `R^2 < R_SQ_MIN` are drawn hollow: there the
rate `m / 2` is a poor summary of `C(t)`.

In [ ]:
R_SQ_MIN = 0.8

colors = Perso_colormap(np.linspace(0.9, 0.1, len(ratio_list)))

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.6), constrained_layout=True)
for j, (ratio, c) in enumerate(zip(ratio_list, colors)):
    good = R_sq_arr[:, j] >= R_SQ_MIN
    for ax, arr in ((axes[0], lyap_arr), (axes[1], end_otoc_arr)):
        ax.plot(epsilon_0_arr, arr[:, j], "-", color=c, lw=2,
                label=rf"$\varepsilon_d = {ratio:g}\,g_2$")
        ax.plot(epsilon_0_arr[good], arr[good, j], "o", color=c, ms=5)
        ax.plot(epsilon_0_arr[~good], arr[~good, j], "o", mfc="white", mec=c, ms=5)
axes[0].set_ylabel(r"$m/2$")
axes[1].set_ylabel(r"$C(T)$")
axes[1].set_yscale("log")
for ax in axes:
    ax.set_xlabel(r"$\varepsilon_p$" + " (rad)")
    ax.set_title(r"$n_a=$" + str(n_a) + ", " + r"$n_b=$" + str(n_b)
                 + f", {heatmap_date} (hollow: $R^2 < {R_SQ_MIN}$)", fontsize=10)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8, ncol=2)
plt.show()

## The rate `m / 2` against the two-photon coupling `g2`

The same `m / 2`, against `g2 = E_J sin(epsilon_0) phi_a^2 phi_b` (article definition), shown as `g2 / 2pi` in MHz.
The curves are labelled by the run ratio `r = epsilon_d / g2`. Hollow points have `R^2 < R_SQ_MIN`, as above.


In [ ]:
g2_list = g2_of(epsilon_0_arr.astype(float)) / (2 * np.pi) * 1e3   # g2 / 2pi, MHz
assert np.all(np.diff(g2_list) > 0)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.6), constrained_layout=True)

im = axes[0].pcolormesh(G2_MESH, RATIO_MESH, lyap_arr, cmap=Perso_colormap)
axes[0].set_ylabel(r"$\varepsilon_d / g_2$")
fig.colorbar(im, ax=axes[0], label=r"$m/2$")

for j, (ratio, c) in enumerate(zip(ratio_list, colors)):
    good = R_sq_arr[:, j] >= R_SQ_MIN
    axes[1].plot(g2_list, lyap_arr[:, j], "-", color=c, lw=2,
                 label=rf"$\varepsilon_d = {ratio:g}\,g_2$")
    axes[1].plot(g2_list[good], lyap_arr[good, j], "o", color=c, ms=5)
    axes[1].plot(g2_list[~good], lyap_arr[~good, j], "o", mfc="white", mec=c, ms=5)
axes[1].axhline(0, color="0.5", lw=0.8)
axes[1].set_ylabel(r"$m/2$")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8, ncol=2)

for ax in axes:
    ax.set_xlabel(r"$g_2 / 2\pi$ (MHz)")
    ax.set_title(r"$m/2$, $n_a=$" + str(n_a) + ", " + r"$n_b=$" + str(n_b)
                 + f", {heatmap_date}", fontsize=10)
plt.show()

## Poster figure: `m / 2` with the classical chaos boundary and the harmonic-balance threshold

The last figure of `plot_OTOCS.ipynb`, with the data above. `y = epsilon_d / 2 g2`
(`= |alpha|^2`), so the heatmap rows are `ratio_list / 2`. On top:

- dashed grey: the onset of classical chaos, the first `epsilon_d / 2 g2` with a
  positive Lyapunov exponent in `data_lyapunov_12_10_2026.json` (`lyapunov_heatmap.ipynb`).
- solid white: the harmonic-balance (Hill) instability of the period-2 orbit,
  `g2_list_12_10_2026_Hill_unstable.json` from `Hill_method_doubling_corrected.ipynb`.
  It is ordered along the ascending `epsilon_d / g2` of `eps_d_ratio_list_12_10_2026_Hill_unstable.json`,
  so it is plotted **without** `[::-1]`.

The axes are clipped to the OTOC data, which cover `epsilon_d / 2 g2` from 0.75 to 4.75,
so the lines are cut there. The inset is `C(t)` of the two examples.

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


# Classical chaos boundary (origin start, lyapunov_heatmap.ipynb): for each g2, the first eps_d/2g2 with lambda > 0
with open(output_dir / "data_lyapunov_12_10_2026.json") as f:
    data_lyapunov = np.array(json.load(f))
with open(output_dir / "g2_list_12_10_2026.json") as f:
    g2_list_bis = np.array(json.load(f)) / 2 / np.pi * 1e3   # MHz
with open(output_dir / "alpha_sq_list_12_10_2026.json") as f:
    alpha_sq = np.array(json.load(f))

alpha_sq_chaos = np.array([alpha_sq[np.where(row == 1)[0][0]] if np.any(row == 1) else np.nan
                           for row in data_lyapunov])

# Harmonic-balance instability of the period-2 orbit.
with open(output_dir / "g2_list_12_10_2026_Hill_unstable.json") as f:
    g2_unstable = np.array(json.load(f), dtype=float)       # MHz
with open(output_dir / "eps_d_ratio_list_12_10_2026_Hill_unstable.json") as f:
    alpha_sq_Hill = np.array(json.load(f)) / 2

# examples: epsilon_d = 10 g2 in the run
g2_example = {name: g2_of(eps) / (2 * np.pi) * 1e3
              for name, eps in (("otoc_example", 0.93), ("otoc_example_stable", 0.2))}
alpha_sq_example = {name: 10 / 2 for name in ("otoc_example", "otoc_example_stable")}

fig, ax = plt.subplots(figsize=(10, 4))

# Heatmap
c = ax.pcolormesh(G2_MESH, RATIO_MESH / 2, lyap_arr, cmap=Perso_colormap)
ax.plot(g2_list_bis, alpha_sq_chaos, ls="--", color="lightgrey",
        label="classical chaos (Lyapunov)")
ax.plot(g2_unstable, alpha_sq_Hill, color="white", lw=1.5,
        label="harmonic-balance instability")
ax.set_xlim(G2_MESH.min(), G2_MESH.max())
ax.set_ylim(RATIO_MESH.min() / 2, RATIO_MESH.max() / 2)

# Labels
ax.set_xlabel(r"$g_2/2\pi$ (MHz)", fontsize=18)
ax.set_ylabel(r"$\varepsilon_d/2g_2$", fontsize=18)
ax.tick_params(axis="both", labelsize=12)
ax.legend(loc="upper left", fontsize=10, frameon=False, labelcolor="white")

# Colorbar
cbar = fig.colorbar(c, ax=ax)
cbar.set_label("Max. Lyap. Exp", fontsize=18)
cbar.ax.tick_params(labelsize=12)

# ---------------- Inset ----------------
axins = inset_axes(ax, width="100%", height="100%",
                   bbox_to_anchor=(0.12, 0.25, 0.4, 0.4),
                   bbox_transform=ax.transAxes)

for name in ("otoc_example", "otoc_example_stable"):
    t, _ = time_axis(len(examples[name]))
    axins.plot(t * kappa, examples[name],
               label=rf"$g_2={g2_example[name]:.0f}$ MHz, $\varepsilon_d/2g_2 = {alpha_sq_example[name]:.2f}$")

axins.set_xlabel(r"$\kappa_b t$", fontsize=18, color="white")
axins.set_ylabel(r"$\mathcal{C}(t)$", fontsize=18, color="white")
axins.tick_params(axis="both", labelsize=12)
axins.tick_params(colors="white", labelcolor="white")
axins.legend(fontsize=8.8)
# --------------------------------------

plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/"
            f"Lyapunov_from_OTOC_init_coherent_{heatmap_date}.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Heatmap
c = ax.pcolormesh(G2_MESH, RATIO_MESH / 2, R_sq_arr, cmap=Perso_colormap)
ax.plot(g2_list_bis, alpha_sq_chaos, ls="--", color="lightgrey",
        label="classical chaos (Lyapunov)")
ax.plot(g2_unstable, alpha_sq_Hill, color="white", lw=1.5,
        label="harmonic-balance instability")
ax.set_xlim(G2_MESH.min(), G2_MESH.max())
ax.set_ylim(RATIO_MESH.min() / 2, RATIO_MESH.max() / 2)

# Labels
ax.set_xlabel(r"$g_2/2\pi$ (MHz)", fontsize=18)
ax.set_ylabel(r"$\varepsilon_d/2g_2$", fontsize=18)
ax.tick_params(axis="both", labelsize=12)
ax.legend(loc="upper left", fontsize=10, frameon=False, labelcolor="white")

# Colorbar
cbar = fig.colorbar(c, ax=ax)
cbar.set_label(r"$R^2$", fontsize=18)
cbar.ax.tick_params(labelsize=12)

# --------------------------------------

plt.show()